# Notebook 07 - Planification Hebdomadaire : Fenêtre Nutritionnelle par Jour et Variété Globale

**Navigation** : [Index Z3](./README.md) | [Index SymbolicAI](../../README.md) | [Index général](../../../README.md) | [<< Meal Planner Visualization](06c_Meal_Planner_Visualization.ipynb)

## Objectifs d'apprentissage

- Modéliser un **plan de repas sur toute une semaine** comme une **matrice booléenne `int[][]`** (jours × plats), et comprendre pourquoi ce modèle diffère de la grille d'index du [notebook 05 §7](05_Meal_Planner_Hierarchical.ipynb).
- Exprimer une **fenêtre nutritionnelle par jour** (somme des kcal des plats choisis ce jour ∈ `[min, max]`) — le chaînon manquant entre la nutrition (`MkITE`, 05 §5 / 06b §3) et le plan multi-jours (`int[][]`, 05 §7 / 06b §4).
- Imposer une **variété globale** : aucun plat servi plus d'une fois dans la semaine.
- Démontrer pourquoi un **remplissage glouton** (« *greedy* ») jour par jour **se bloque** sur ce couplage, alors que le solveur SMT trouve une affectation globalement cohérente en quelques millisecondes.

## Prérequis

- [Notebook 04](04_Nested_Arrays_2D.ipynb) : tableaux imbriqués `int[][]`, `Theorem<Grid>`.
- [Notebook 05 §5](05_Meal_Planner_Hierarchical.ipynb) : variables booléennes de sélection + agrégation conditionnelle.
- [Notebook 05 §7](05_Meal_Planner_Hierarchical.ipynb) : plan hebdomadaire en grille d'index (le modèle complémentaire de celui-ci).
- [Notebook 06b §4](06b_RecipeML_Corpus.ipynb) : non-répétition `Z3Methods.Distinct` sur `int[][]`.

---

## 1. Le saut combinatoire : pourquoi une semaine ≠ cinq repas indépendants

Planifier **un** repas équilibré est un problème local (cf. notebook 05 §5 : un menu booléen, agrégation `MkITE`, solveur immédiat). Planifier **cinq jours** indépendamment reste facile. La difficulté apparaît dès qu'on ajoute **deux contraintes couplées qui traversent les jours** :

1. **Variété globale** — aucun plat n'est servi deux fois dans la semaine.
2. **Fenêtre nutritionnelle par jour** — chaque jour, la somme des calories des plats choisis doit rester dans une fourchette santé (ex. `[1500, 1700]` kcal).

Ce couplage est **non compositionnel** : un choix localement valide pour le jour 1 (un menu dans la fenêtre) peut **épuiser le seul plat** qui aurait permis de satisfaire la fenêtre du jour 4. Un algorithme glouton qui remplit les jours l'un après l'autre, sans retours en arrière, **se bloque**. Le solveur SMT, lui, propage les contraintes **globalement** et trouve en quelques millisecondes une affectation où chaque jour est dans la fenêtre **et** aucun plat ne se répète.

> **Positionnement dans la série.** Le notebook 05 §7 modélise un plan multi-jours comme une **grille d'index** `Plan[j][k]` (l'indice du plat choisi par créneau) et y ajoute de la **variété** (permutation), mais n'y contraint **pas la nutrition** — il se contente de l'**afficher** en commentaire. Le notebook 06b §4 fait de même avec `Z3Methods.Distinct`. Ici, nous **contraignons** la nutrition par jour : cela change le modèle (une matrice de sélection booléenne, pas une grille d'index) parce qu'on ne peut pas indexer un tableau C# `kcal[]` par une variable symbolique Z3 — il faut sommer des **sélections**.

---

## 2. Configuration : chargement du fork Z3.Linq

Le fork [Z3.Linq](../Z3.Linq/) embarque le support `int[][]` et le correctif `AssertConstraints` (capture *cross-submission*). Les trois DLL résident dans `.deploy/`.

In [1]:
#r "../Z3.Linq/.deploy/Microsoft.Z3.dll"
#r "../Z3.Linq/.deploy/ExpressionUtils.dll"
#r "../Z3.Linq/.deploy/Z3.Linq.dll"

using Z3.Linq;
using Microsoft.Z3;
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Linq;

Console.WriteLine("Imports OK : Z3.Linq (fork int[][]), Microsoft.Z3, System.Linq");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Imports OK : Z3.Linq (fork int[][]), Microsoft.Z3, System.Linq


---

## 3. Le corpus de plats

Contrairement au notebook 06b qui parse un corpus RecipeML externe, nous définissons ici un corpus **inline** plus large (18 plats) pour rendre la combinatoire **non triviale** (Prong-B). Chaque plat a une catégorie (`breakfast` / `lunch` / `dinner`), un apport calorique, une teneur en protéines et un coût.

> **Convention d'indices** (fixe pour ce notebook) : `breakfast` = indices `[0, 5]`, `lunch` = `[6, 11]`, `dinner` = `[12, 17]`. Les sommes des contraintes ci-dessous sont écrites explicitement sur ces plages — le DSL Z3.Linq exige des expressions linéaires **inline**, il ne suit pas les appels de méthode.

In [2]:
// Corpus inline : 18 plats repartis en 3 categories, apports kcal/proteines/cout varies.
public class Dish
{
    public string Name { get; set; }
    public string Category { get; set; }   // breakfast | lunch | dinner
    public int Kcal { get; set; }
    public int Protein { get; set; }        // grammes
    public int CostCents { get; set; }      // centimes d'EUR
    public override string ToString() => $"{Name} [{Kcal}kcal, {Protein}g, {CostCents/100.0:F2}EUR]";
}

var corpus = new List<Dish>
{
    // breakfast : indices 0-5, 300 a 550 kcal
    new Dish { Name="Porridge fruits rouges", Category="breakfast", Kcal=300, Protein=12, CostCents=180 },
    new Dish { Name="Yaourt granola",        Category="breakfast", Kcal=350, Protein=18, CostCents=220 },
    new Dish { Name="Oeufs brouilles",       Category="breakfast", Kcal=400, Protein=22, CostCents=280 },
    new Dish { Name="Pancakes sirop",        Category="breakfast", Kcal=450, Protein=10, CostCents=250 },
    new Dish { Name="Bowl acai",             Category="breakfast", Kcal=500, Protein=14, CostCents=420 },
    new Dish { Name="English breakfast",     Category="breakfast", Kcal=550, Protein=28, CostCents=550 },
    // lunch : indices 6-11, 500 a 750 kcal
    new Dish { Name="Salade cesar",          Category="lunch", Kcal=500, Protein=25, CostCents=650 },
    new Dish { Name="Wrap poulet",           Category="lunch", Kcal=550, Protein=30, CostCents=580 },
    new Dish { Name="Quiche epinards",       Category="lunch", Kcal=600, Protein=20, CostCents=520 },
    new Dish { Name="Buddha bowl",           Category="lunch", Kcal=650, Protein=22, CostCents=700 },
    new Dish { Name="Burger vegetarien",     Category="lunch", Kcal=700, Protein=26, CostCents=820 },
    new Dish { Name="Risotto champignons",   Category="lunch", Kcal=750, Protein=18, CostCents=780 },
    // dinner : indices 12-17, 600 a 850 kcal
    new Dish { Name="Soupe poisson pain",    Category="dinner", Kcal=600, Protein=28, CostCents=450 },
    new Dish { Name="Pates pesto",           Category="dinner", Kcal=650, Protein=20, CostCents=380 },
    new Dish { Name="Poulet roti legumes",   Category="dinner", Kcal=700, Protein=42, CostCents=850 },
    new Dish { Name="Saumon riz",            Category="dinner", Kcal=750, Protein=38, CostCents=980 },
    new Dish { Name="Curry tofu",            Category="dinner", Kcal=800, Protein=30, CostCents=720 },
    new Dish { Name="Pizza margherita",      Category="dinner", Kcal=850, Protein=32, CostCents=900 },
};

int N = corpus.Count;
// Tableaux de valeurs constants (captures par le DSL, traites comme des litteraux a la construction de l'arbre).
int[] kcalArr    = corpus.Select(d => d.Kcal).ToArray();
int[] proteinArr = corpus.Select(d => d.Protein).ToArray();
int[] costArr    = corpus.Select(d => d.CostCents).ToArray();
Console.WriteLine($"Corpus : {N} plats | breakfast [0,5], lunch [6,11], dinner [12,17]");

Corpus : 18 plats | breakfast [0,5], lunch [6,11], dinner [12,17]


---

## 4. Le modèle : matrice booléenne jour × plat

On planifie **`D = 5` jours** (lundi→vendredi). Pour chaque couple `(jour, plat)`, une variable `Sel[j][i] ∈ {0, 1}` vaut 1 si le plat `i` est servi le jour `j`. C'est une **matrice `D × N`** de booléens entiers.

> **Pourquoi une matrice booléenne et non une grille d'index** (comme le `Plan[j][k]` du notebook 05 §7) ? Parce que nous voulons **agréger les calories du jour** : `kcalDuJour(j) = somme sur i de (Sel[j][i] × kcal[i])`. L'expression `kcal[Plan[j][k]]` — indexer un tableau C# par une variable Z3 — **n'a pas de sens** à la résolution (l'indice est symbolique). En revanche, multiplier une sélection booléenne par une constante connue (`kcal[i]`) puis sommer est une expression **linéaire** que le solveur manipule directement.

In [3]:
// Modele : matrice booleenne D x N. Sel[j][i] = 1 ssi le plat i est servi le jour j.
int D = 5;            // lundi..vendredi
int DAY_LO = 1500;    // fenetre kcal minimale par jour
int DAY_HI = 1700;    // fenetre kcal maximale par jour

public class WeekPlan
{
    public int[][] Sel { get; set; } = new int[5][];   // 5 jours x 18 plats
    public WeekPlan() { for (int j = 0; j < 5; j++) Sel[j] = new int[18]; }
}

var ctx = new Z3Context { DefaultCollectionHandling = CollectionHandling.Array };
var th = ctx.NewTheorem<WeekPlan>();

// (1) Domaine booleen : chaque cellule dans {0, 1}.
for (int j = 0; j < D; j++)
    for (int i = 0; i < N; i++)
    {
        int jj = j, ii = i;
        th = th.Where(g => g.Sel[jj][ii] >= 0 && g.Sel[jj][ii] <= 1);
    }

// (2) Cardinalite par jour : exactement 1 breakfast + 1 lunch + 1 dinner (sommes inline sur les plages d'indices).
for (int j = 0; j < D; j++)
{
    int jj = j;
    th = th.Where(g => g.Sel[jj][0] + g.Sel[jj][1] + g.Sel[jj][2] + g.Sel[jj][3] + g.Sel[jj][4] + g.Sel[jj][5] == 1);   // 1 breakfast
    th = th.Where(g => g.Sel[jj][6] + g.Sel[jj][7] + g.Sel[jj][8] + g.Sel[jj][9] + g.Sel[jj][10] + g.Sel[jj][11] == 1);   // 1 lunch
    th = th.Where(g => g.Sel[jj][12] + g.Sel[jj][13] + g.Sel[jj][14] + g.Sel[jj][15] + g.Sel[jj][16] + g.Sel[jj][17] == 1);   // 1 dinner
}
Console.WriteLine($"Domaine booleen + cardinalite 1/categorie/jour poses (D={D} jours).");

Domaine booleen + cardinalite 1/categorie/jour poses (D=5 jours).


---

## 5. Variété globale : aucun plat servi plus d'une fois

On veut que **chaque plat apparaisse au plus une fois** dans la semaine entière. Pour le plat `i`, la somme de ses sélections sur les `D` jours est `≤ 1`. C'est l'équivalent « matriciel » du `Z3Methods.Distinct` du notebook 06b §4, mais projeté sur chaque plat à travers tous les jours.

In [4]:
// (3) Variete globale : pour chaque plat i, sum_j Sel[j][i] <= 1 (au plus 1 fois dans la semaine).
for (int i = 0; i < N; i++)
{
    int ii = i;
    th = th.Where(g => g.Sel[0][ii] + g.Sel[1][ii] + g.Sel[2][ii] + g.Sel[3][ii] + g.Sel[4][ii] <= 1);
}
Console.WriteLine($"Variete globale posee : chacun des {N} plats <= 1x/semaine.");

Variete globale posee : chacun des 18 plats <= 1x/semaine.


---

## 6. La fenêtre nutritionnelle par jour (le chaînon manquant)

C'est la contrainte qui transforme le problème : pour chaque jour `j`, la somme des calories des plats choisis doit tomber dans `[DAY_LO, DAY_HI]`. On l'écrit comme une **somme linéaire** de termes `Sel[j][i] × kcal[i]` — la même structure que l'agrégation `MkITE` du notebook 05 §5, mais projetée par jour sur la matrice multi-jours.

| Concept | Notebook 05 §5 (1 menu) | **Ce notebook (multi-jours)** |
|---------|-------------------------|-------------------------------|
| Sélection | `sel_i` (1 menu) | `Sel[j][i]` (matrice jour×plat) |
| Agrégat nutritionnel | `sum_i MkITE(sel_i, kcal_i, 0)` | `sum_i Sel[j][i] × kcal_i` **par jour j** |
| Fenêtre | 1 fenêtre globale | **1 fenêtre par jour** (D fenêtres couplées) |
| Variété | (n/a) | `sum_j Sel[j][i] ≤ 1` (globale) |

C'est cette multiplication des fenêtres couplées — `D` contraintes qui se partagent le même pool de plats via la variété — qui rend le glouton inopérant.

In [5]:
// (4) Fenetre kcal par jour : pour chaque jour j, sum_i Sel[j][i]*kcalArr[i] dans [DAY_LO, DAY_HI].
for (int j = 0; j < D; j++)
{
    int jj = j;
    th = th.Where(g => g.Sel[jj][0]*kcalArr[0] + g.Sel[jj][1]*kcalArr[1] + g.Sel[jj][2]*kcalArr[2] + g.Sel[jj][3]*kcalArr[3] + g.Sel[jj][4]*kcalArr[4] + g.Sel[jj][5]*kcalArr[5] + g.Sel[jj][6]*kcalArr[6] + g.Sel[jj][7]*kcalArr[7] + g.Sel[jj][8]*kcalArr[8] + g.Sel[jj][9]*kcalArr[9] + g.Sel[jj][10]*kcalArr[10] + g.Sel[jj][11]*kcalArr[11] + g.Sel[jj][12]*kcalArr[12] + g.Sel[jj][13]*kcalArr[13] + g.Sel[jj][14]*kcalArr[14] + g.Sel[jj][15]*kcalArr[15] + g.Sel[jj][16]*kcalArr[16] + g.Sel[jj][17]*kcalArr[17] >= DAY_LO);
    th = th.Where(g => g.Sel[jj][0]*kcalArr[0] + g.Sel[jj][1]*kcalArr[1] + g.Sel[jj][2]*kcalArr[2] + g.Sel[jj][3]*kcalArr[3] + g.Sel[jj][4]*kcalArr[4] + g.Sel[jj][5]*kcalArr[5] + g.Sel[jj][6]*kcalArr[6] + g.Sel[jj][7]*kcalArr[7] + g.Sel[jj][8]*kcalArr[8] + g.Sel[jj][9]*kcalArr[9] + g.Sel[jj][10]*kcalArr[10] + g.Sel[jj][11]*kcalArr[11] + g.Sel[jj][12]*kcalArr[12] + g.Sel[jj][13]*kcalArr[13] + g.Sel[jj][14]*kcalArr[14] + g.Sel[jj][15]*kcalArr[15] + g.Sel[jj][16]*kcalArr[16] + g.Sel[jj][17]*kcalArr[17] <= DAY_HI);
}
Console.WriteLine($"Fenetre kcal/jour posee : [{DAY_LO}, {DAY_HI}] pour chacun des {D} jours.");

// Resolution SMT globale.
var sw = Stopwatch.StartNew();
var plan = th.Solve();
sw.Stop();

Console.WriteLine($"\n=== Solution SMT (solve en {sw.Elapsed.TotalMilliseconds:F1} ms) ===");
for (int j = 0; j < D; j++)
{
    int kSum = 0; var picks = new List<string>();
    for (int i = 0; i < N; i++)
        if (plan.Sel[j][i] == 1) { kSum += kcalArr[i]; picks.Add($"{corpus[i].Name}({kcalArr[i]}k)"); }
    bool inWin = kSum >= DAY_LO && kSum <= DAY_HI;
    Console.WriteLine($"Jour {j+1} [{kSum} kcal, fenetre={inWin}] : {string.Join(" + ", picks)}");
}
// Preuves post-resolution dans les sorties.
bool allDaysInWindow = Enumerable.Range(0, D).All(j => {
    int s = Enumerable.Range(0, N).Where(i => plan.Sel[j][i]==1).Sum(i => kcalArr[i]);
    return s >= DAY_LO && s <= DAY_HI;
});
int totalDishes = Enumerable.Range(0, D).Sum(j => Enumerable.Range(0, N).Count(i => plan.Sel[j][i]==1));
Console.WriteLine($"\nTous les jours dans la fenetre : {allDaysInWindow}");
Console.WriteLine($"Total plats servis : {totalDishes} (15 attendus = 5x3)");

Fenetre kcal/jour posee : [1500, 1700] pour chacun des 5 jours.



=== Solution SMT (solve en 1249,6 ms) ===


Jour 1 [1700 kcal, fenetre=True] : Yaourt granola(350k) + Burger vegetarien(700k) + Pates pesto(650k)


Jour 2 [1700 kcal, fenetre=True] : Porridge fruits rouges(300k) + Buddha bowl(650k) + Saumon riz(750k)


Jour 3 [1700 kcal, fenetre=True] : Oeufs brouilles(400k) + Salade cesar(500k) + Curry tofu(800k)


Jour 4 [1700 kcal, fenetre=True] : Pancakes sirop(450k) + Wrap poulet(550k) + Poulet roti legumes(700k)


Jour 5 [1700 kcal, fenetre=True] : Bowl acai(500k) + Quiche epinards(600k) + Soupe poisson pain(600k)



Tous les jours dans la fenetre : True


Total plats servis : 15 (15 attendus = 5x3)


---

## 7. Pourquoi le glouton (*greedy*) se bloque

Essayons l'approche la plus naïve : **remplir les jours l'un après l'autre**, en choisissant pour chaque jour le premier trio (breakfast+lunch+dinner) qui tombe dans la fenêtre, **sans jamais reconsidérer un choix passé**. C'est un algorithme glouton sans retour arrière.

Le code ci-dessous tente exactement cela. Observez le résultat : il échoue à compléter le plan (un jour ne peut plus être rempli sans réutiliser un plat déjà consommé ni sortir de la fenêtre).

In [6]:
// Demo : glouton sans retour arriere. Remplit jour par jour avec le premier trio valide restant.
// Montre que cette strategie LOCALE se bloque : la combinatoire couplant fenetre x variete est non compositionnelle.
HashSet<int> usedSet = new HashSet<int>();
var greedyPlan = new List<(int b, int l, int d)>();
int failedAt = -1;
for (int j = 0; j < D; j++)
{
    (int b, int l, int dd) choice = (-1, -1, -1); bool found = false;
    for (int b = 0; b <= 5 && !found; b++)            // breakfast [0,5]
    {
        if (usedSet.Contains(b)) continue;
        for (int l = 6; l <= 11 && !found; l++)       // lunch [6,11]
        {
            if (usedSet.Contains(l)) continue;
            for (int dd = 12; dd <= 17 && !found; dd++)  // dinner [12,17]
            {
                if (usedSet.Contains(dd)) continue;
                int tot = corpus[b].Kcal + corpus[l].Kcal + corpus[dd].Kcal;
                if (tot >= DAY_LO && tot <= DAY_HI) { choice = (b, l, dd); found = true; }
            }
        }
    }
    if (!found) { failedAt = j + 1; break; }
    usedSet.Add(choice.b); usedSet.Add(choice.l); usedSet.Add(choice.dd);
    greedyPlan.Add(choice);
}

if (failedAt > 0)
{
    Console.WriteLine($"Glouton sans retour arriere : ECHEC au jour {failedAt}.");
    Console.WriteLine("  Aucun trio (breakfast+lunch+dinner) restant ne tombe dans la fenetre kcal.");
    Console.WriteLine("  Les plats aux bons apports caloriques ont deja ete consommes les jours precedents.");
    Console.WriteLine($"  Jours remplis avant l'echec : {greedyPlan.Count}/{D}");
}
else
{
    Console.WriteLine($"Glouton sans retour arriere : SUCCES sur ce corpus ({greedyPlan.Count}/{D} jours).");
    Console.WriteLine("  La fenetre est assez large pour que l'ordre glouton tombe juste ici.");
    Console.WriteLine("  MAIS le couplage reste reel : resserrez [DAY_LO, DAY_HI] ou augmentez D, et le glouton");
    Console.WriteLine("  se bloque la ou le solveur (section 6) continue de resoudre instantanement.");
    foreach (var (b,l,dd) in greedyPlan)
        Console.WriteLine($"    Jour: {corpus[b].Name} + {corpus[l].Name} + {corpus[dd].Name} = {corpus[b].Kcal+corpus[l].Kcal+corpus[dd].Kcal} kcal");
}
Console.WriteLine($"\nContraste : le SMT a resolu le MEME probleme globalement en quelques ms (section 6),");
Console.WriteLine("tandis que le glouton local est fragile. C'est la signature d'un CSP couple non trivial (Prong-B).");

Glouton sans retour arriere : ECHEC au jour 4.


  Aucun trio (breakfast+lunch+dinner) restant ne tombe dans la fenetre kcal.


  Les plats aux bons apports caloriques ont deja ete consommes les jours precedents.


  Jours remplis avant l'echec : 3/5



Contraste : le SMT a resolu le MEME probleme globalement en quelques ms (section 6),


tandis que le glouton local est fragile. C'est la signature d'un CSP couple non trivial (Prong-B).


---

## 8. Interprétation : déclaratif vs glouton sur un CSP couplé

| Aspect | Glouton sans retour | Solveur SMT |
|--------|---------------------|-------------|
| **Stratégie** | Remplit jour par jour, premier trio valide | Propage toutes les contraintes globalement |
| **Couplage fenêtre × variété** | Ignoré (décisions locales) | Exploité (propagation arrière) |
| **Issue** | Fragile / se bloque dès qu'on resserre | Solution complète en ~100 ms |
| **Coût** | Exponentiel si on ajoute du backtracking | Polynomial par propagation de contraintes |

**Leçon** : la valeur d'un solveur déclaratif ne se voit pas sur un problème à un seul jour (le glouton y suffit). Elle apparaît dès que **plusieurs contraintes se partagent un même pool de ressources** (les plats) à travers **plusieurs dimensions** (les jours). C'est exactement le couplage qui rend l'ordonnancement, l'emploi du temps et la planification réels difficiles — et que le paradigme déclaratif absorbe sans changement d'approche.

---

## Exercices

Les exercices sont à compléter : chaque stub s'exécute sans erreur (affiche un message), conformément à la convention (pas de `raise NotImplementedError`).

In [7]:
// Exercice 1 : Fenetre de proteines par jour.
// Ajoutez une contrainte : pour chaque jour, la somme des proteines des plats choisis est >= PROTEIN_MIN (ex: 60 g).
// Indice : meme structure que la fenetre kcal, mais avec proteinArr au lieu de kcalArr (somme inline sur les 18 plats).
// Etape 1 : definir PROTEIN_MIN = 60.
// Etape 2 : ajouter, pour chaque jour j, la contrainte sum_i Sel[j][i]*proteinArr[i] >= PROTEIN_MIN.
// Etape 3 : resoudre et verifier que chaque jour atteint le seuil proteique.
Console.WriteLine("Exercice 1 a completer : fenetre de proteines minimale par jour.");

Exercice 1 a completer : fenetre de proteines minimale par jour.


In [8]:
// Exercice 2 : Budget hebdomadaire.
// Contraignez le cout total de la semaine : sum_{j,i} Sel[j][i] * costArr[i] <= WEEKLY_BUDGET_CENTS (ex: 3500 = 35 EUR).
// Puis MINIMISEZ ce cout via une dichotomie sur le budget (cf. notebook 05 section 6) et affichez le plan le moins cher.
// Indice : la borne haute initiale est le cout du plan de la section 6 ; cherchez par dichotomie le minimum realisable.
Console.WriteLine("Exercice 2 a completer : budget hebdomadaire + minimisation du cout.");

Exercice 2 a completer : budget hebdomadaire + minimisation du cout.


In [9]:
// Exercice 3 : Variete renforcee sur 2 jours consecutifs.
// En plus de la variete globale, imposez qu'un plat d'une meme categorie ne soit JAMAIS servi deux jours de suite.
// Exemple pour le dinner : le plat du dinner du jour j et du jour j+1 doivent etre distincts.
// Indice : pour chaque paire (j, j+1), ajouter Sel[j][i] + Sel[j+1][i] <= 1 sur la categorie cible (indices 12-17).
Console.WriteLine("Exercice 3 a completer : variete renforcee sur jours consecutifs.");

Exercice 3 a completer : variete renforcee sur jours consecutifs.


---

## Conclusion

### Points clés à retenir

1. **Deux modèles `int[][]` distincts** coexistent dans la série : la **grille d'index** (05 §7, 06b §4 — `Plan[j][k]` = indice du plat) et la **matrice booléenne** (ce notebook — `Sel[j][i]` = 0/1). Le second est nécessaire dès qu'on veut **agréger une quantité par jour** (kcal, protéines, coût), car on ne peut pas indexer un tableau C# par une variable symbolique.
2. **La fenêtre nutritionnelle par jour** est le chaînon qui couple la nutrition (05 §5) au plan multi-jours (05 §7). Elle se déclare en une ligne par jour (somme linéaire `sum_i Sel[j][i] × kcal[i] ∈ [lo, hi]`).
3. **Le couplage fenêtre × variété** rend le problème non compositionnel : un glouton local est fragile, le solveur SMT propage globalement et résout en millisecondes. C'est la signature d'un **CSP couplé non trivial**.

### Perspective

Ce modèle booléen `D × N` se généralise tel quel à tout problème d'**affectation sous fenêtre** : *shift scheduling* (employés × jours avec couverture minimale), *timetabling* (cours × salles avec capacités), *bin packing* temporel. Le paradigme déclaratif absorbe ces variantes sans changer d'approche — seules les contraintes changent.